# ASDEA_sensors -- all plots (data 17:00-18:00)

A fully functional gallery: it loads the desktop folder, takes the window
between 5 pm and 6 pm, and runs every method showing its inputs and its plot.
The package is imported directly (installed in your environment).

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"
T0, T1 = "2026-05-31 17:00:00", "2026-05-31 18:00:00"   # 5 pm to 6 pm
DEV = "MOF00135"

## Sensor geometry (for the building plots)

In [ ]:
settings.SENSOR_GEOMETRY["MOF00134"].update(E=500000.0, N=6300000.0, elev=0.0)
settings.SENSOR_GEOMETRY["MNAT0031"].update(E=500000.0, N=6300000.0, elev=7.0)
settings.SENSOR_GEOMETRY["MNAT0034"].update(E=500000.0, N=6300000.0, elev=10.5)
settings.SENSOR_GEOMETRY["MOF00135"].update(E=500000.0, N=6300000.0, elev=14.0)
settings.SENSOR_GEOMETRY["MOF00136"].update(E=500006.0, N=6300004.0, elev=14.0)

## Load and take the 5-6 pm window, then the pipeline

In [ ]:
ds = SensorDataset(DATA, verbose=True)
W   = ds.device(DEV).get_window(T0, T1)          # 5 pm - 6 pm
WF  = W.baseline().filter(0.1, 24.9)             # corrected + band-passed
WFD = WF.derive()                                # vel and disp available
sig = WFD.signal(components="all")
print("window samples:", sig.n, " dt:", round(sig.dt, 6), " duration:", round(sig.duration, 1), "s")

## 1. Time signals (acceleration, velocity, displacement)

In [ ]:
from asdea_sensors.plotting import signal_plots
signal_plots.plot_signals(sig, components="all", kind="acc")
signal_plots.plot_signals(sig, components="all", kind="vel")
signal_plots.plot_signals(sig, components="all", kind="disp")

## 2. Fourier amplitude spectrum (raw and Konno-Ohmachi)

In [ ]:
from asdea_sensors.plotting import fourier_plots
f_raw = WF.fourier(component="x", num_frequencies=6, prominence=1e-6, distance_frac=0.02, smooth=None)
fourier_plots.plot_fourier(f_raw, component="x")
f_kon = WF.fourier(component="x", num_frequencies=6, smooth="konno", bexp=40)
fourier_plots.plot_fourier(f_kon, component="x", smooth="konno")
print("dominant freqs (Hz):", [round(float(x), 2) for x in f_raw["dom_freqs"]])

## 3. Newmark response spectra (PSa, PSv, Sd)

In [ ]:
from asdea_sensors.plotting import newmark_plots
spec = WF.newmark(component="x", zeta=0.05, max_period=5.01, dT=0.01, factor=1.0)
newmark_plots.plot_newmark(spec, component="x", quantity="PSa")
newmark_plots.plot_newmark(spec, component="x", quantity="PSv")
newmark_plots.plot_newmark(spec, component="x", quantity="Sd")

## 4. RotD spectra (0, 50, 100)

In [ ]:
from asdea_sensors.plotting import rotd_plots
r = WF.rotd(comp_x="x", comp_y="y", rotd=50, angle_step=15, max_period=3.0, dT=0.02)
r0  = WF.rotd(rotd=0,   angle_step=15, max_period=3.0, dT=0.02)
r100 = WF.rotd(rotd=100, angle_step=15, max_period=3.0, dT=0.02)
merged = {"T": r["T"], "ROTD0": r0["ROTD0"], "ROTD50": r["ROTD50"], "ROTD100": r100["ROTD100"]}
rotd_plots.plot_rotd(merged, rotd=(0, 50, 100))

## 5. Arias intensity (Husid)

In [ ]:
from asdea_sensors.plotting import arias_plots
a = dict(WF.arias(component="x", low=5, high=95)); a["dt"] = sig.dt
arias_plots.plot_arias(a, component="x")
print("CAV =", round(float(WF.cav(component="x")["CAV"]), 5),
      " | Housner SI =", round(float(WF.housner(component="x", T1=0.1, T2=2.5)["SI"]), 6))
print("peaks:", {ax: {k: round(float(v), 5) for k, v in WFD.peaks(component="all")[ax].items()} for ax in ["x","y","z"]})

## 6. PSD and band energy across sensors

In [ ]:
from asdea_sensors.plotting import psd_plots
p = WF.psd(component="x", nperseg=512, noverlap=256, bands=[(0,1),(1,2.5),(2.5,5),(5,10)])
psd_plots.plot_psd(p, component="x")

from asdea_sensors.seismic import psd as psd_mod
by_dev = {}
for d in ds.devices:
    s = ds.device(d).get_window(T0, T1).baseline().filter(0.1, 24.9).signal(components="x")
    by_dev[d] = psd_mod.compute(s.acc_x, s.dt, bands=[(0,1),(1,2.5),(2.5,5),(5,10)])
psd_plots.plot_psd_bands(by_dev, band=(1, 2.5))

## 7. STFT spectrogram

In [ ]:
from asdea_sensors.plotting import stft_plots
s = WF.stft(component="x", nperseg=256, noverlap=224, fmax=25.0)
stft_plots.plot_stft(s, component="x")

## 8. Building -- transfer function and coherence (floor 4 / base)

In [ ]:
from asdea_sensors.building import transfer_function, coherence
from asdea_sensors.plotting import transfer_plots
top  = ds.device("MOF00135").get_window(T0, T1).baseline().filter(0.1, 24.9).signal(components="x")
base = ds.device("MOF00134").get_window(T0, T1).baseline().filter(0.1, 24.9).signal(components="x")
n = min(top.n, base.n)
frf = transfer_function.compute(top.acc_x[:n], base.acc_x[:n], top.dt, smooth="konno", fmax=25.0)
transfer_plots.plot_transfer_function(frf)
coh = coherence.compute(top.acc_x[:n], base.acc_x[:n], top.dt)
plt.figure(figsize=(9,3.5)); plt.plot(coh["f"], coh["coherence"])
plt.xlim(0,25); plt.xlabel("Frequency [Hz]"); plt.ylabel("Coherence [-]")
plt.title("Coherence MOF00135 / MOF00134 - x", fontweight="bold"); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("modal freqs (Hz):", [round(float(x),2) for x in frf["modal_freqs"][:5]])

## 9. Building -- modal tracking over the hour

In [ ]:
from asdea_sensors.plotting import modal_plots
mt = WF.modal_tracking(component="x", window="10min", overlap=0.5, fband=(1.0, 8.0), n_modes=2)
modal_plots.plot_modal_tracking(mt)

## 10. Building -- floor-4 torsion and interstory drift

In [ ]:
from asdea_sensors.building import torsion, geometry, drift
a4 = ds.device("MOF00135").get_window(T0, T1).baseline().filter(0.1, 24.9).signal(components="x")
b4 = ds.device("MOF00136").get_window(T0, T1).baseline().filter(0.1, 24.9).signal(components="x")
n = min(a4.n, b4.n)
dist = geometry.plan_distance(settings.SENSOR_GEOMETRY, "MOF00135", "MOF00136")
theta = torsion.floor_rotation(a4.acc_x[:n], b4.acc_x[:n], dist)
tspec = torsion.torsional_spectrum(theta, a4.dt, smooth=None)
plt.figure(figsize=(9,3)); plt.plot(np.arange(theta.size)*a4.dt, theta)
plt.xlabel("Time [s]"); plt.ylabel("Rotation [rad]"); plt.title("Floor-4 torsion theta(t)", fontweight="bold")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("torsional freq:", round(float(tspec["torsional_freq"]),2), "Hz  | plan distance:", round(dist,2), "m")

up = ds.device("MOF00135").get_window(T0, T1).baseline().filter(0.1, 24.9).derive().signal(components="x")
lo = ds.device("MNAT0034").get_window(T0, T1).baseline().filter(0.1, 24.9).derive().signal(components="x")
n = min(up.n, lo.n)
dr = drift.interstory_drift(up.disp_x[:n], lo.disp_x[:n], story_height=3.5)
plt.figure(figsize=(9,3)); plt.plot(np.arange(n)*up.dt, dr["drift"])
plt.xlabel("Time [s]"); plt.ylabel("Relative disp [m]"); plt.title("Interstory drift p4 - p3", fontweight="bold")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("max drift:", round(float(dr["max_drift"]),6), "m  | max ratio:", round(float(dr["max_ratio"]),6))

## 11. Ambient -- STA/LTA, windows, mean spectrum

In [ ]:
from asdea_sensors.plotting import ambient_plots

# WF is a windowed device handle; condition it and run the ambient analysis.
amb = WF.baseline().ambient(component="x")
ambient_plots.plot_sta_lta(amb)
ambient_plots.plot_spectrum(amb, min_freq=0.3)
print("dominant period:", round(1.0 / amb["f_dom"], 3), "s")

## 12. Ambient -- HVSR (Nakamura) and amplification

In [ ]:
from asdea_sensors.ambient import hvsr, amplification
hv = hvsr.compute(sig.acc_x, sig.acc_y, sig.acc_z, config, combine="geometric")
plt.figure(figsize=(9,3.5)); plt.plot(hv["freqs"], hv["HV"])
plt.xlim(0,25); plt.xlabel("Frequency [Hz]"); plt.ylabel("H/V [-]")
plt.title("HVSR (Nakamura) - MOF00135  f0=%.2f Hz" % hv["f0"], fontweight="bold")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

ref = ds.device("MOF00134").get_window(T0, T1).baseline().filter(0.1, 24.9).signal(components="x")
others = {d: ds.device(d).get_window(T0, T1).baseline().filter(0.1, 24.9).signal(components="x").acc_x
          for d in ["MNAT0031", "MNAT0034", "MOF00135"]}
amp = amplification.compute(ref.acc_x, others, ref.dt, basis="fourier", config=config)
from asdea_sensors.plotting import amplification_plots
amplification_plots.plot_amplification(amp)